<a href="https://colab.research.google.com/github/thiendang184-droid/Robot-AI-homework/blob/main/Bai_23_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install osmnx networkx folium

import osmnx as ox
import networkx as nx
import folium
import numpy as np


center = [10.770, 106.677]
depots = [[10.775, 106.690], [10.765, 106.665]] # Kho A, Kho B

np.random.seed(42)
#Giả lập 8 khách hàng
customers = (np.random.normal(size=(8, 2)) * 0.01 + center).tolist()

#TẢI MẠNG LƯỚI GIAO THÔNG (Bán kính 3km)
print("Đang tải mạng lưới đường bộ (vui lòng đợi vài giây)...")
G = ox.graph_from_point(center, dist=3000, network_type='drive')

def get_node(loc):
    return ox.distance.nearest_nodes(G, X=loc[1], Y=loc[0])

def get_real_route(node_a, node_b):
    try:
        length = nx.shortest_path_length(G, node_a, node_b, weight='length')
        path = nx.shortest_path(G, node_a, node_b, weight='length')
        return length, path
    except nx.NetworkXNoPath:
        return float('inf'), []

depot_nodes = [get_node(d) for d in depots]
customer_nodes = [get_node(c) for c in customers]

unopt_dist = 0
unopt_route_coords = []
curr_node = depot_nodes[0]
#PHƯƠNG ÁN KHÔNG TỐI ƯU
for c_node in customer_nodes:
    dist, path = get_real_route(curr_node, c_node)
    unopt_dist += dist
    if path:
        unopt_route_coords.extend([(G.nodes[n]['y'], G.nodes[n]['x']) for n in path])
    curr_node = c_node

dist, path = get_real_route(curr_node, depot_nodes[0])
unopt_dist += dist
if path:
    unopt_route_coords.extend([(G.nodes[n]['y'], G.nodes[n]['x']) for n in path])

#PHƯƠNG ÁN TỐI ƯU

assigned_A, assigned_B = [], []
for c_node in customer_nodes:
    distA, _ = get_real_route(c_node, depot_nodes[0])
    distB, _ = get_real_route(c_node, depot_nodes[1])
    if distA < distB:
        assigned_A.append(c_node)
    else:
        assigned_B.append(c_node)

def optimize_route(start_node, nodes_to_visit):
    if not nodes_to_visit: return 0, []
    total_dist = 0
    full_path_coords = []
    unvisited = nodes_to_visit.copy()
    curr = start_node

    while unvisited:

        next_node = min(unvisited, key=lambda x: get_real_route(curr, x)[0])
        dist, path = get_real_route(curr, next_node)
        total_dist += dist
        if path:
            full_path_coords.extend([(G.nodes[n]['y'], G.nodes[n]['x']) for n in path])
        curr = next_node
        unvisited.remove(curr)


    dist, path = get_real_route(curr, start_node)
    total_dist += dist
    if path:
        full_path_coords.extend([(G.nodes[n]['y'], G.nodes[n]['x']) for n in path])

    return total_dist, full_path_coords

dist_A, route_A_coords = optimize_route(depot_nodes[0], assigned_A)
dist_B, route_B_coords = optimize_route(depot_nodes[1], assigned_B)
opt_dist = dist_A + dist_B

#IN ĐÁNH GIÁ KẾT QUẢ

print("-" * 50)
print(f"Tổng quãng đường KHÔNG TỐI ƯU: {unopt_dist/1000:.2f} km")
print(f"Tổng quãng đường TỐI ƯU (AI):  {opt_dist/1000:.2f} km")
savings = (unopt_dist - opt_dist) / unopt_dist * 100
print(f"=> Thuật toán AI giúp TIẾT KIỆM: {savings:.2f}% quãng đường lái xe thực tế.")
print("-" * 50)

m = folium.Map(location=center, zoom_start=14)

if unopt_route_coords:
    folium.PolyLine(unopt_route_coords, color='gray', weight=3, opacity=0.4, popup="Tuyến gốc").add_to(m)

if route_A_coords:
    folium.PolyLine(route_A_coords, color='red', weight=5, opacity=0.8, popup="Xe từ Kho A").add_to(m)
if route_B_coords:
    folium.PolyLine(route_B_coords, color='blue', weight=5, opacity=0.8, popup="Xe từ Kho B").add_to(m)

folium.Marker(depots[0], popup="Kho A", icon=folium.Icon(color='red', icon='warehouse', prefix='fa')).add_to(m)
folium.Marker(depots[1], popup="Kho B", icon=folium.Icon(color='blue', icon='warehouse', prefix='fa')).add_to(m)

for i, c in enumerate(customers):
    folium.CircleMarker(c, radius=6, color='black', fill=True, fill_color='white', fill_opacity=1, popup=f"Khách {i+1}").add_to(m)

display(m)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.6 MB/s eta 0:00:00
Đang tải mạng lưới đường bộ (vui lòng đợi vài giây)...
--------------------------------------------------
Tổng quãng đường KHÔNG TỐI ƯU: 24.07 km
Tổng quãng đường TỐI ƯU (AI):  15.80 km
=> Thuật toán AI giúp TIẾT KIỆM: 34.38% quãng đường lái xe thực tế.
--------------------------------------------------
